# LeRobot Data: Download, Convert, Train, and Evaluate

This notebook walks through the full workflow for using [LeRobot](https://github.com/huggingface/lerobot) datasets with VLA Foundry: downloading a dataset from HuggingFace Hub, converting it to WebDataset format, training a diffusion policy model, and running inference. We set the training scale to a small number to make the tutorial but this is not enough to get a meaningful policy. The user is welcome to modify the tutorial as they see fit.

We use `lerobot/pusht` as the running example throughout this tutorial. PushT is a simple 2D pushing task that makes a great starting point.

## Prerequisites

- Python 3.12+ with VLA Foundry installed via `uv` (see [Installation](../README.md#installation))
- The **`Python (vla_foundry)`** Jupyter kernel — install it once from the repo root:
  ```bash
  bash tutorials/install_kernel.sh
  ```
  This syncs dependencies (preprocessing, gym-pusht, etc.) and registers the kernel.
- Restart your notebook kernel after installing the vla_foundry kernel and select it for execution.
- For training, a machine with one or more GPUs
- (Optional) AWS credentials configured if you want to read/write data on S3

**Storage:** This tutorial downloads ~12MB of LeRobot data from HuggingFace and generates ~2GB of preprocessed WebDataset shards. Ensure you have at least 3GB of free disk space before starting.


In [ ]:
import os
import subprocess

os.environ["CUDA_VISIBLE_DEVICES"] = "0"

# Change to the repo root (works regardless of where the notebook is opened from)
repo_root = subprocess.check_output(["git", "rev-parse", "--show-toplevel"], text=True).strip()
os.chdir(repo_root)
print(f"Working directory: {os.getcwd()}")

**Stop here.** Switch to the **"Python (vla_foundry)"** kernel in your notebook UI (Kernel → Change Kernel) and restart it before running the cells below.

## 1. Download a LeRobot Dataset

### Browsing Available Datasets

LeRobot datasets are hosted on HuggingFace Hub. You can browse them at:
- [https://huggingface.co/lerobot](https://huggingface.co/lerobot)

Some well-known datasets include:
- `lerobot/pusht` -- Push-T task (2D)
- `lerobot/aloha_sim_transfer_cube_human` -- ALOHA bimanual cube transfer (simulation)
- `lerobot/aloha_sim_insertion_human` -- ALOHA peg insertion (simulation)
- `lerobot/xarm_lift_medium_replay` -- xArm lift task

### Downloading with huggingface-cli

> **Important -- Use `--revision v2.0`.** The default version on HuggingFace is now v3.0, which uses a different directory layout that the converter does not support yet. Always specify `--revision v2.0` when downloading datasets for use with VLA Foundry.

> ⚠️ **Heavy download & preprocessing ahead.** The next cells download `lerobot/pusht` (~1 GB) from HuggingFace and produce ~2 GB of preprocessed WebDataset shards. Both steps are idempotent (safe to re-run; skips files already on disk). Under `jupyter nbconvert`, pass `--ExecutePreprocessor.timeout=1800` or run the download from a terminal first.


In [ ]:
from huggingface_hub import snapshot_download

snapshot_download(
    "lerobot/pusht",
    repo_type="dataset",
    revision="v2.0",
    local_dir="./tutorials/data/lerobot/pusht",
)

### LeRobot Data Format (v2.0)

After downloading, the dataset directory has this structure (v2.0 format):

```
pusht/
  meta/
    info.json              # Dataset metadata (fps, features, shapes, etc.)
    episodes.jsonl         # Per-episode metadata (length, task, index)
    tasks.jsonl            # Task descriptions (language instructions)
  data/
    chunk-000/
      episode_000000.parquet   # Episode data (states, actions)
      episode_000001.parquet
      ...
  videos/                  # (Optional) Video files for camera observations
    chunk-000/
      observation.image/
        episode_000000.mp4
        ...
```

Key files:
- **`meta/info.json`**: Contains the dataset FPS, feature names, data shapes, and number of episodes
- **`meta/episodes.jsonl`**: One JSON object per line, each describing an episode
- **`data/chunk-XXXX/episode_XXXXXX.parquet`**: Parquet files containing observation and action data
- **`videos/`**: (Optional) Video files organized by camera name and chunk

## 2. Convert to WebDataset Format

VLA Foundry uses WebDataset tar shards for training. The conversion script handles LeRobot datasets with `--type lerobot`.

### Inspecting Your Dataset

Before running preprocessing, inspect the dataset to discover the correct column names for observations, actions, and images. Column names vary between datasets, so always check before proceeding.

In [ ]:
import pandas as pd

df = pd.read_parquet("./tutorials/data/lerobot/pusht/data/chunk-000/episode_000000.parquet")
print("Columns:", df.columns.tolist())
print("\nFirst few rows:")
df.head()

**For video-based datasets** (like `pusht`), image/camera columns will **not** appear in the parquet file. Instead, camera names come from the subdirectories under `videos/`:

In [ ]:
import os

# For video-based datasets, camera names come from the videos/ directory
video_dirs = os.listdir("./tutorials/data/lerobot/pusht/videos/chunk-000/")
print("Camera names:", video_dirs)

### Basic Conversion (Local)

The example below shows the preprocessing command for converting a LeRobot dataset to webdataset tar shards.

Arguments that accept lists use Python list literal syntax as a string (e.g., `"['observation.state']"`).

### Key Arguments

| Argument | Description |
|---|---|
| `--type "lerobot"` | Selects the LeRobot converter |
| `--source_episodes` | List of paths to LeRobot dataset root directories (local or S3) |
| `--output_dir` | Where to write the converted WebDataset shards (local or S3) |
| `--camera_names` | List of camera/image names (e.g., `observation.image`) |
| `--observation_keys` | List of observation column names to include (e.g., `observation.state`) |
| `--action_keys` | List of action column names (e.g., `action` or `actions`) |
| `--config_path` | Path to a YAML file with windowing/preprocessing parameters |
| `--samples_per_shard` | Number of training samples per tar shard (default: 128) |

### Understanding the Preprocessing Config

The preprocessing config at `vla_foundry/config_presets/data/robotics_preprocessing_params_1past_14future.yaml` defines how raw episodes are windowed into training samples. Key parameters:

- **`past_lowdim_steps: 1`, `future_lowdim_steps: 14`** -- Each training sample contains a window of 16 timesteps: 1 past frame + the current frame + 14 future frames. The name "1past_14future" reflects this. Actions and proprioception are stored for all 16 timesteps.
- **`image_indices: [-1, 0]`** -- Two images are extracted per sample: one at timestep -1 (the past frame) and one at timestep 0 (the current/anchor frame). This gives the model a sense of motion.
- **`resize_images_size: [384, 384]`** -- Images are resized to 384x384 pixels using center cropping (`image_resizing_method: center_crop`).
- **`stride: 1`** -- Sliding window moves by 1 timestep, so consecutive frames generate overlapping samples, maximizing data utilization.
- **`padding_strategy: copy`** -- At episode boundaries, missing frames are filled by copying the nearest available frame (up to `max_padding_left: 3` and `max_padding_right: 15`).

**Timing:** On a single GPU machine, data conversion takes approximately 2-3 minutes for the PushT dataset (~25k frames across 206 episodes).

In [ ]:
import os

output_dir = os.path.abspath("./tutorials/data/preprocessed/pusht/")
source_dir = os.path.abspath("./tutorials/data/lerobot/pusht/")

# Point Ray workers to the real venv so uv doesn't create a new one (avoids build race condition)
os.environ["UV_PROJECT_ENVIRONMENT"] = os.path.abspath(".venv")

!MPLBACKEND= uv run --group preprocessing python vla_foundry/data/preprocessing/preprocess_robotics_to_tar.py \
    --type "lerobot" \
    --source_episodes "['{source_dir}']" \
    --output_dir {output_dir} \
    --camera_names "['observation.image']" \
    --observation_keys "['observation.state']" \
    --action_keys "['action']" \
    --samples_per_shard 100 \
    --config_path "vla_foundry/config_presets/data/robotics_preprocessing_params_1past_14future.yaml"

### Output Structure

After conversion, the output directory contains:

```
tutorials/data/preprocessed/pusht/
  shards/
    manifest.jsonl         # Manifest listing all shards and their sample counts
    stats.json             # Dataset statistics (mean, std, min, max per field)
    shard_000000.tar       # WebDataset tar shards
    shard_000001.tar
    ...
  episodes/
    manifest.jsonl         # Episode-based manifest
    stats.json             # Same statistics (copy)
    episode_*.tar          # Episode-based tar shards
  frames/
    manifest.jsonl         # Frame-level manifest
    frame_*.tar            # Per-timestep frame metadata
```

What each directory contains:
- **`shards/`** -- The main training data. Each tar file is a WebDataset shard containing images (JPEG) and low-dimensional data (NPZ files with actions, proprioception, and masks) for a batch of training samples. The `manifest.jsonl` and `stats.json` here are referenced directly by the training config.
- **`episodes/`** -- Per-episode tar archives containing episode-level metadata such as language instructions and episode boundaries. Used for episode-aware sampling and evaluation.
- **`frames/`** -- Per-timestep frame metadata used for windowed sampling. Contains the mapping from frame indices back to episodes, enabling the sliding-window sample construction.

The key files for training are:
- **`shards/manifest.jsonl`** -- referenced by `dataset_manifest` in the training config
- **`shards/stats.json`** -- referenced by `dataset_statistics` in the training config

### Additional Options

**Ray Parallelization** -- For large datasets, add `--ray_address local --ray_num_cpus 16` to parallelize episode processing across multiple CPU cores.

**Writing to S3** -- Replace `--output_dir ./tutorials/data/preprocessed/pusht/` with an S3 path like `--output_dir s3://my-bucket/vla_foundry_datasets/lerobot/pusht/`.

**Multi-Camera** -- For datasets with multiple cameras, list all camera names: `--camera_names "['cam_left', 'cam_right', 'cam_wrist']"`.

## 3. Train a Model

After preprocessing, you can train a model using the converted data. VLA Foundry uses a YAML config system powered by draccus.

### Creating a Training Config

Create a YAML config file that references your converted data. Here is an example for training a diffusion policy on the PushT dataset.

**Note:** The `<<: !include` syntax is a draccus feature that merges in fields from another YAML file. This lets you reuse common presets for models, data pipelines, and hyperparameters.

> **Important -- Temporal config must match between preprocessing and training.**
> The `past_lowdim_steps` and `future_lowdim_steps` values in your preprocessing config must match the `lowdim_past_timesteps` and `lowdim_future_timesteps` in the training data config.

### Available Model Presets

Model presets are located in `vla_foundry/config_presets/models/`. Some commonly used ones:

| Preset | Description |
|---|---|
| `diffusion_policy.yaml` | Diffusion/flow matching policy with CLIP vision backbone |
| `transformer_tiny.yaml` | Tiny transformer (for testing/debugging) |
| `transformer_100m.yaml` | 100M parameter transformer |
| `vlm_3b.yaml` | 3B vision-language model (PaliGemma) |
| `vla_diffusion_paligemma2.yaml` | VLA diffusion policy with PaliGemma2 backbone |

In [ ]:
%%writefile pusht_training_config.yaml
# Training config for PushT diffusion policy with a small ViT backbone (no pretrained weights).
save_path: tutorials/checkpoints/pusht
# Uses 96x96 images directly (native PushT resolution), no augmentation.

model:
  <<: !include vla_foundry/config_presets/models/diffusion_policy.yaml
  vision_language_backbone:
    type: vit_backbone
    img_size: 96
    hidden_dim: 64
    inter_dim: 256
    n_heads: 4
    n_layers: 4
    patch_size: 8
  # Minimal denoising head (this IS the diffusion head, not an LLM)
  transformer:
    type: transformer
    hidden_dim: 64
    n_heads: 4
    n_layers: 2
    is_causal: True

data:
  <<: !include vla_foundry/config_presets/data/diffusion_policy.yaml
  processor: "none"
  image_size: 96
  dataset_manifest:
    - ./tutorials/data/preprocessed/pusht/shards/manifest.jsonl
  dataset_statistics:
    - ./tutorials/data/preprocessed/pusht/shards/stats.json
  dataset_modality:
    - robotics
  dataset_weighting:
    - 1.0
  camera_names:
    - observation.image
  image_indices:
    - -1
    - 0
  action_fields:
    - action
  proprioception_fields:
    - observation.state
  num_workers: 4

distributed:
  fsdp: True

hparams:
  <<: !include vla_foundry/config_presets/hparams/diffusion_policy.yaml
  per_gpu_batch_size: 32
  global_batch_size: 512

### Running Training Locally

Launch training with `torchrun`. Key training arguments:

| Argument | Description |
|---|---|
| `--config_path` | Path to training config YAML |
| `--total_train_samples` | Total number of training samples to process |
| `--num_checkpoints` | Number of checkpoints to save during training |
| `--remote_sync` | S3 path to sync checkpoints to (optional) |
| `--wandb False` | Disable Weights & Biases logging |
| `--name` | Experiment name (used for logging) |

For this tutorial, we run a small training job (5000 samples, 1 checkpoint) with wandb disabled.

**Timing:** On a single GPU, training 5000 samples with this small model takes approximately 10-15 seconds. A production-scale run (100k+ samples with a larger model) will take significantly longer.

In [ ]:
!uv run torchrun --nproc_per_node=1 --nnodes=1 vla_foundry/main.py \
  --config_path pusht_training_config.yaml \
  --total_train_samples 5000 \
  --num_checkpoints 1 \
  --wandb False

### Additional Training Options

**Previewing the resolved config** -- Before training, preview the fully resolved configuration with all `!include` directives expanded:
```bash
uv run torchrun --nproc_per_node=1 --nnodes=1 vla_foundry/main.py \
  --config_path pusht_training_config.yaml \
  --total_train_samples 100000 \
  --num_checkpoints 5 \
  --resolve_configs True \
  --resolve_configs_path ./
```

**Syncing checkpoints to S3** -- Add `--remote_sync s3://my-bucket/model_checkpoints/pusht_diffusion_policy` to save checkpoints to S3 during training.

**Finetuning from pretrained weights** -- Use `--model.resume_from_checkpoint <path_to_checkpoint.pt>` and `--model.resume_weights_only True` to load only model weights without resuming optimizer state.

## 4. Verify Inference

Now let's verify the trained model works by loading the checkpoint, feeding it a sample from the preprocessed data, and running a forward pass to generate action predictions.

In [ ]:
import glob
import os

# Find the experiment directory (most recent one)
experiment_dirs = sorted(glob.glob("tutorials/checkpoints/pusht/*/"), key=os.path.getmtime)
assert len(experiment_dirs) > 0, "No experiment directories found. Did training complete?"
experiment_dir = experiment_dirs[-1].rstrip("/")
print(f"Using experiment directory: {experiment_dir}")

# Find the checkpoint
checkpoint_files = sorted(glob.glob(os.path.join(experiment_dir, "checkpoints", "checkpoint_*.pt")))
assert len(checkpoint_files) > 0, "No checkpoints found."
checkpoint_path = checkpoint_files[-1]
print(f"Using checkpoint: {checkpoint_path}")

# Config path
config_path = os.path.join(experiment_dir, "config.yaml")
print(f"Using config: {config_path}")

# Data shard
shard_path = "tutorials/data/preprocessed/pusht/shards/shard_000000.tar"
print(f"Using data shard: {shard_path}")

In [ ]:
import io
import json
import tarfile

import numpy as np
import torch
from PIL import Image

from vla_foundry.data.processor.robotics_processor import RoboticsProcessor
from vla_foundry.file_utils import load_model_checkpoint
from vla_foundry.models import create_model
from vla_foundry.params.train_experiment_params import load_experiment_params_from_yaml

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")

# Load experiment config
cfg = load_experiment_params_from_yaml(config_path, localize_params=True)
print(f"Model type: {cfg.model.type}")
print(f"Action dim: {cfg.model.action_dim}")
print(f"Proprioception dim: {cfg.model.proprioception_dim}")

# Create model and load checkpoint
model = create_model(cfg.model, load_pretrained=False)
start_ckpt, global_step, _ = load_model_checkpoint(model, checkpoint_path)
print(f"Checkpoint loaded: checkpoint_num={start_ckpt}, global_step={global_step}")

model.to(DEVICE)
model.eval()
print(f"Model on {DEVICE}, eval mode")

In [ ]:
# Load RoboticsProcessor for normalization
robotics_processor = RoboticsProcessor.from_pretrained(experiment_dir)
print("RoboticsProcessor loaded")
print(f"  Normalizer enabled: {robotics_processor.normalizer is not None}")

# Extract one sample from the tar shard
sample_files = {}
first_sample_key = None

with tarfile.open(shard_path, "r") as tar:
    for member in tar.getmembers():
        name = member.name
        sample_key = name.split(".", 1)[0]
        if first_sample_key is None:
            first_sample_key = sample_key
        if sample_key != first_sample_key:
            if len(sample_files) >= 4:
                break
            continue
        f = tar.extractfile(member)
        if f is not None:
            sample_files[name] = f.read()

print(f"\nSample key: {first_sample_key}")
print(f"Files in sample: {list(sample_files.keys())}")

# Parse the sample data
images = {}
lowdim_data = None
metadata = None
language_instructions = None

for filename, data in sample_files.items():
    if filename.endswith(".jpg") or filename.endswith(".png"):
        img = Image.open(io.BytesIO(data)).convert("RGB")
        images[filename] = img
        print(f"  Image: {filename} -> size={img.size}")
    elif filename.endswith(".npz"):
        lowdim_data = dict(np.load(io.BytesIO(data)))
        print(f"  Lowdim fields: {list(lowdim_data.keys())}")
        for k, v in lowdim_data.items():
            print(f"    {k}: shape={v.shape}, dtype={v.dtype}")
    elif filename.endswith("metadata.json"):
        metadata = json.loads(data.decode())
        print(f"  Metadata: {metadata}")
    elif filename.endswith("language_instructions.json"):
        language_instructions = json.loads(data.decode())
        print(f"  Language instructions: {language_instructions}")

In [ ]:
import torchvision.transforms as T

# Build model input and run inference
lowdim_past_timesteps = cfg.data.lowdim_past_timesteps
lowdim_future_timesteps = cfg.data.lowdim_future_timesteps
total_timesteps = lowdim_past_timesteps + 1 + lowdim_future_timesteps
action_dim = cfg.data.action_dim
image_names = cfg.data.image_names

print("Data config:")
print(f"  past_timesteps: {lowdim_past_timesteps}")
print(f"  future_timesteps: {lowdim_future_timesteps}")
print(f"  total_timesteps: {total_timesteps}")
print(f"  action_dim: {action_dim}")
print(f"  image_names: {image_names}")

# Collect images in order and convert to tensors
img_to_tensor = T.ToTensor()

ordered_image_tensors = []
for img_name in image_names:
    for filename, img in images.items():
        if img_name in filename:
            # Resize to model's expected size (96x96)
            img_resized = img.resize((96, 96))
            ordered_image_tensors.append(img_to_tensor(img_resized))
            break

print(f"\nNumber of images for model: {len(ordered_image_tensors)}")

# Stack into [1, N, C, H, W]
pixel_values = torch.stack(ordered_image_tensors).unsqueeze(0)
# Dummy text inputs (ViT backbone ignores them)
input_ids = torch.zeros(1, 1, dtype=torch.long)
attention_mask = torch.ones(1, 1, dtype=torch.long)

print(f"  pixel_values: {pixel_values.shape}")

# Build action and proprioception tensors
raw_actions = torch.tensor(lowdim_data["action"], dtype=torch.float32)
if robotics_processor.normalizer is not None:
    actions_normalized = robotics_processor.normalizer.normalize_tensor(raw_actions.unsqueeze(0), "action")
else:
    actions_normalized = raw_actions.unsqueeze(0)

raw_proprio = torch.tensor(lowdim_data["observation.state"], dtype=torch.float32)
if robotics_processor.normalizer is not None:
    proprio_normalized = robotics_processor.normalizer.normalize_tensor(raw_proprio.unsqueeze(0), "observation.state")
else:
    proprio_normalized = raw_proprio.unsqueeze(0)
proprio_normalized = proprio_normalized[:, : lowdim_past_timesteps + 1]

# Build past_mask
past_mask = torch.zeros(1, total_timesteps, dtype=torch.bool)
past_mask[:, : lowdim_past_timesteps + 1] = True

# Move to device
input_ids = input_ids.to(DEVICE)
attention_mask = attention_mask.to(DEVICE)
pixel_values = pixel_values.to(DEVICE, dtype=torch.float32)
actions_normalized = actions_normalized.to(DEVICE, dtype=torch.float32)
past_mask = past_mask.to(DEVICE)
proprio_normalized = proprio_normalized.to(DEVICE, dtype=torch.float32)

# Run inference
print("\nRunning generate_actions...")
NUM_FLOW_STEPS = 10

with torch.no_grad():
    predicted_actions = model.generate_actions(
        input_ids=input_ids,
        pixel_values=pixel_values,
        actions=actions_normalized,
        attention_mask=attention_mask,
        num_inference_steps=NUM_FLOW_STEPS,
        past_mask=past_mask,
        proprioception=proprio_normalized,
    )

print(f"\nPredicted actions shape: {predicted_actions.shape}")
print(f"Predicted actions dtype: {predicted_actions.dtype}")

pred_cpu = predicted_actions.cpu().float().numpy()
print("\nPredicted actions stats (normalized):")
print(f"  Min: {pred_cpu.min():.6f}")
print(f"  Max: {pred_cpu.max():.6f}")
print(f"  Mean: {pred_cpu.mean():.6f}")
print(f"  Std: {pred_cpu.std():.6f}")

print("\nFirst 5 predicted action timesteps (normalized):")
for t in range(min(5, pred_cpu.shape[1])):
    print(f"  t={t}: {pred_cpu[0, t]}")

# Denormalize to get actual action values
if robotics_processor.normalizer is not None:
    denorm_actions = robotics_processor.normalizer.denormalize_tensor(predicted_actions.cpu(), "action").numpy()
    print("\nDenormalized actions (first 5 timesteps):")
    for t in range(min(5, denorm_actions.shape[1])):
        print(f"  t={t}: {denorm_actions[0, t]}")
    print(f"\n  Denormalized range: [{denorm_actions.min():.2f}, {denorm_actions.max():.2f}]")

print("\nInference verification complete!")

## 5. Interactive PushT Evaluation

Now let's close the loop: run the trained model inside the actual PushT simulation environment. This requires the `gym-pusht` package (a Gymnasium environment for the 2D Push-T task).

The eval loop:
1. Resets the environment and gets the initial observation (96×96 image + 2D agent position)
2. At each step, builds the model input from the live observation (two images, proprioception, text instruction)
3. Runs `generate_actions` to get an action chunk (16 timesteps)
4. Executes the **first predicted future action** in the environment
5. Records frames for visualization

> **Note:** The model was only trained for 5000 samples in this tutorial, so don't expect great performance. For a well-performing policy, train for 100k+ samples.

In [ ]:
import gym_pusht  # noqa: F401 -- registers the gymnasium environment
import gymnasium as gym
import numpy as np
import torch
from PIL import Image
from torchvision import transforms

# --- Configuration ---
MAX_STEPS = 300  # Maximum environment steps per episode
NUM_FLOW_STEPS = 10  # Denoising steps for action generation
ACTION_EXEC_HORIZON = 8  # How many actions to execute before re-planning
IMG_SIZE = 96  # Native PushT resolution, matches training

# Data config from training
lowdim_past_timesteps = cfg.data.lowdim_past_timesteps  # 1
lowdim_future_timesteps = cfg.data.lowdim_future_timesteps  # 14
total_timesteps = lowdim_past_timesteps + 1 + lowdim_future_timesteps  # 16
anchor_idx = lowdim_past_timesteps  # index of "current" timestep in the window

# Pre-compute static tensors
past_mask = torch.zeros(1, total_timesteps, dtype=torch.bool, device=DEVICE)
past_mask[:, : lowdim_past_timesteps + 1] = True
dummy_actions = torch.zeros(1, total_timesteps, cfg.data.action_dim, device=DEVICE)

# Image transform: uint8 HWC -> float32 CHW [0, 1]
img_transform = transforms.ToTensor()  # handles both conversion and normalization to [0,1]


def obs_to_pixel_values(prev_pixels: np.ndarray, curr_pixels: np.ndarray) -> torch.Tensor:
    """Convert two observation images to model input [1, 2, C, H, W]."""
    t_prev = img_transform(prev_pixels)  # [C, H, W]
    t_curr = img_transform(curr_pixels)  # [C, H, W]
    return torch.stack([t_prev, t_curr]).unsqueeze(0)  # [1, 2, C, H, W]


# --- Create environment ---
env = gym.make("gym_pusht/PushT-v0", obs_type="pixels_agent_pos", render_mode="rgb_array")
obs, info = env.reset(seed=42)

prev_pixels = obs["pixels"]
curr_pixels = obs["pixels"]
prev_agent_pos = obs["agent_pos"].copy()
curr_agent_pos = obs["agent_pos"].copy()

frames = [env.render()]
rewards = []
action_queue = []

print(f"Environment created. Running eval for up to {MAX_STEPS} steps...")
print(f"  Image size: {obs['pixels'].shape} (native, no resizing)")
print(f"  Agent pos: {obs['agent_pos']}")
print()

for step in range(MAX_STEPS):
    if len(action_queue) == 0:
        # Build proprioception: [t-1, t0] normalized
        proprio_raw = torch.tensor(
            np.stack([prev_agent_pos, curr_agent_pos]),
            dtype=torch.float32,
        ).unsqueeze(0)  # [1, 2, 2]
        if robotics_processor.normalizer is not None:
            proprio_norm = robotics_processor.normalizer.normalize_tensor(proprio_raw, "observation.state")
        else:
            proprio_norm = proprio_raw

        # Build pixel values directly (no CLIP processor needed)
        pixel_values = obs_to_pixel_values(prev_pixels, curr_pixels).to(DEVICE, dtype=torch.float32)

        # Generate actions (no input_ids/attention_mask — ViT backbone ignores them)
        with torch.no_grad():
            predicted_actions = model.generate_actions(
                input_ids=torch.zeros(1, 1, dtype=torch.long, device=DEVICE),
                pixel_values=pixel_values,
                actions=dummy_actions,
                attention_mask=torch.ones(1, 1, dtype=torch.long, device=DEVICE),
                num_inference_steps=NUM_FLOW_STEPS,
                past_mask=past_mask,
                proprioception=proprio_norm.to(DEVICE, dtype=torch.float32),
            )

        # Denormalize and extract future actions
        if robotics_processor.normalizer is not None:
            denorm = robotics_processor.normalizer.denormalize_tensor(predicted_actions.cpu(), "action").numpy()[0]
        else:
            denorm = predicted_actions.cpu().numpy()[0]

        future_actions = denorm[anchor_idx : anchor_idx + ACTION_EXEC_HORIZON]
        action_queue = list(future_actions)

    # Execute next action
    action = action_queue.pop(0)
    action = np.clip(action, 0.0, 512.0).astype(np.float32)

    obs, reward, terminated, truncated, info = env.step(action)
    rewards.append(reward)
    frames.append(env.render())

    # Update observation history
    prev_pixels = curr_pixels
    prev_agent_pos = curr_agent_pos.copy()
    curr_pixels = obs["pixels"]
    curr_agent_pos = obs["agent_pos"].copy()

    if terminated or truncated:
        print(f"Episode ended at step {step + 1}")
        break

    if (step + 1) % 50 == 0:
        print(f"  Step {step + 1}/{MAX_STEPS}  reward={reward:.4f}  coverage={info.get('coverage', 0):.4f}")

env.close()

print(f"\nEval complete: {len(rewards)} steps")
print(f"  Final reward: {rewards[-1]:.4f}")
print(f"  Max reward: {max(rewards):.4f}")
print(f"  Final coverage: {info.get('coverage', 0):.4f}")
print(f"  Success: {info.get('is_success', False)}")
print(f"  Frames captured: {len(frames)}")

In [ ]:
# Render the rollout as an inline video in the notebook
import base64

import imageio.v3 as iio
from IPython.display import HTML, display

video_path = "tutorials/data/pusht_eval_rollout.mp4"
iio.imwrite(video_path, np.stack(frames), fps=10, codec="h264", plugin="pyav")
print(f"Video saved to {video_path}")

# Display inline
with open(video_path, "rb") as f:
    video_data = f.read()
b64 = base64.b64encode(video_data).decode()
display(
    HTML(f"""
<video width="400" controls autoplay loop>
  <source src="data:video/mp4;base64,{b64}" type="video/mp4">
</video>
""")
)

## 6. Deployment

For simulation evaluation of a trained VLA checkpoint, use [`lbm_eval`](https://github.com/ToyotaResearchInstitute/lbm_eval). See `vla_foundry/inference/scripts/README.md` for instructions on launching the policy server.

For deploying a trained model as a policy server (e.g., for real robot control), see `vla_foundry/inference/robotics/inference_policy.py`.

## Tips and Common Issues

- **Camera name mismatch**: The `--camera_names` passed during preprocessing must match the column names in the LeRobot parquet files
- **Action key names vary**: Some LeRobot datasets use `action` (singular) while others use `actions` (plural)
- **Image format**: The converter handles both inline image bytes in parquet files and video-based storage (MP4 files). It auto-detects which format is used
- **Memory usage**: For very large datasets, use Ray parallelization to distribute processing across cores
- **Statistics file**: The `stats.json` file is critical for normalization during training. If preprocessing is interrupted before statistics are saved, re-run

**See also**:
- `tutorials/diffusion_policy.md` -- Diffusion policy (Spartan) data workflow
- `tutorials/adding_new_models.md` -- Adding custom models
- `vla_foundry/data/preprocessing/README.md` -- Ray cluster setup for preprocessing
- `vla_foundry/inference/scripts/README.md` -- inference policy server scripts